In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score
from sklearn.datasets import make_blobs, make_moons

# 0. Создание структуры папок

dirs = ['data', 'artifacts', 'artifacts/figures', 'artifacts/labels']
for d in dirs:
    os.makedirs(d, exist_ok=True)

In [2]:
# 1. Генерация/Загрузка данных (Имитация CSV)
def generate_fake_data():
    # DS1: Разные шкалы + шум
    X1, _ = make_blobs(n_samples=300, centers=3, cluster_std=1.0, random_state=42)
    X1[:, 0] = X1[:, 0] * 1000  # Разная шкала
    df1 = pd.DataFrame(X1, columns=['feat1', 'feat2'])
    df1['noise'] = np.random.normal(0, 500, size=300)
    df1['sample_id'] = [f's1_{i}' for i in range(300)]
    df1.to_csv('data/S07-hw-dataset-01.csv', index=False)

    # DS2: Нелинейность (Луны)
    X2, _ = make_moons(n_samples=300, noise=0.1, random_state=42)
    df2 = pd.DataFrame(X2, columns=['feat1', 'feat2'])
    df2['outlier'] = 10.0 # Добавим мусора
    df2.iloc[0, 2] = 50.0 
    df2['sample_id'] = [f's2_{i}' for i in range(300)]
    df2.to_csv('data/S07-hw-dataset-02.csv', index=False)

    # DS4: Категории + Пропуски
    X4, _ = make_blobs(n_samples=300, centers=4, n_features=5, random_state=42)
    df4 = pd.DataFrame(X4, columns=[f'num_{i}' for i in range(5)])
    df4['cat_feature'] = np.random.choice(['A', 'B', 'C'], size=300)
    df4.loc[0:10, 'num_0'] = np.nan # Пропуски
    df4['sample_id'] = [f's4_{i}' for i in range(300)]
    df4.to_csv('data/S07-hw-dataset-04.csv', index=False)

generate_fake_data()

# Словари для хранения итогов
all_metrics = {}
best_configs = {}

In [3]:
# 2. Обработка датасетов


datasets = {
    'ds1': 'data/S07-hw-dataset-01.csv',
    'ds2': 'data/S07-hw-dataset-02.csv',
    'ds4': 'data/S07-hw-dataset-04.csv'
}

for ds_name, path in datasets.items():
    print(f"\n--- Processing {ds_name} ---")
    df = pd.read_csv(path)
    
    # Первичный анализ
    print(df.info())
    X = df.drop(columns=['sample_id'])
    sample_ids = df['sample_id']
    
    # Препроцессинг
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
    
    num_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    
    preprocessor = ColumnTransformer(transformers=[
        ('num', num_transformer, num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ])
    
    X_pre = preprocessor.fit_transform(X)
    
    # --- Кластеризация 1: KMeans ---
    inertias = []
    silhouettes = []
    k_range = range(2, 11)
    
    for k in k_range:
        km = KMeans(n_clusters=k, n_init=10, random_state=42)
        labels = km.fit_predict(X_pre)
        silhouettes.append(silhouette_score(X_pre, labels))
        
    # График подбора K (Артефакт 1, 2, 3)
    plt.figure()
    plt.plot(k_range, silhouettes, marker='o')
    plt.title(f'Silhouette vs K ({ds_name})')
    plt.savefig(f'artifacts/figures/{ds_name}_k_selection.png')
    plt.close()
    
    best_k = k_range[np.argmax(silhouettes)]
    km_final = KMeans(n_clusters=best_k, n_init=10, random_state=42)
    km_labels = km_final.fit_predict(X_pre)
    
    # --- Кластеризация 2: DBSCAN или Agglomerative ---
    if ds_name == 'ds2': # Для лун берем Agglomerative с разным linkage
        model = AgglomerativeClustering(n_clusters=2, linkage='ward')
        other_labels = model.fit_predict(X_pre)
        other_name = 'Agglomerative'
    else:
        model = DBSCAN(eps=0.5, min_samples=5)
        other_labels = model.fit_predict(X_pre)
        other_name = 'DBSCAN'
        
    # --- Метрики ---
    def get_metrics(data, labels):
        # Для DBSCAN исключаем шум (-1) при расчете метрик
        mask = labels != -1
        if np.sum(mask) < 2 or len(np.unique(labels[mask])) < 2:
            return 0, 0, 0, np.mean(labels == -1)
        return (
            silhouette_score(data[mask], labels[mask]),
            davies_bouldin_score(data[mask], labels[mask]),
            calinski_harabasz_score(data[mask], labels[mask]),
            np.mean(labels == -1)
        )

    m_km = get_metrics(X_pre, km_labels)
    m_ot = get_metrics(X_pre, other_labels)
    
    all_metrics[ds_name] = {
        'KMeans': {'sil': m_km[0], 'db': m_km[1], 'ch': m_km[2]},
        other_name: {'sil': m_ot[0], 'db': m_ot[1], 'ch': m_ot[2], 'noise_ratio': m_ot[3]}
    }
    
    # Выбор лучшего (упрощенно по силуэту)
    best_labels = km_labels if m_km[0] > m_ot[0] else other_labels
    best_configs[ds_name] = "KMeans" if m_km[0] > m_ot[0] else other_name
    
    # --- Визуализация PCA (Артефакт 4, 5, 6) ---
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_pre)
    plt.figure(figsize=(8,6))
    plt.scatter(X_pca[:, 0], X_pca[:, 1], c=best_labels, cmap='viridis', alpha=0.7)
    plt.title(f'PCA Result: {ds_name} (Best: {best_configs[ds_name]})')
    plt.colorbar(label='Cluster')
    plt.savefig(f'artifacts/figures/{ds_name}_pca.png')
    plt.close()
    
    # Сохранение меток
    output_labels = pd.DataFrame({'sample_id': sample_ids, 'cluster_label': best_labels})
    output_labels.to_csv(f'artifacts/labels/labels_hw07_{ds_name}.csv', index=False)


--- Processing ds1 ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   feat1      300 non-null    float64
 1   feat2      300 non-null    float64
 2   noise      300 non-null    float64
 3   sample_id  300 non-null    object 
dtypes: float64(3), object(1)
memory usage: 9.5+ KB
None

--- Processing ds2 ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   feat1      300 non-null    float64
 1   feat2      300 non-null    float64
 2   outlier    300 non-null    float64
 3   sample_id  300 non-null    object 
dtypes: float64(3), object(1)
memory usage: 9.5+ KB
None

--- Processing ds4 ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 7 columns):
 #   Column       Non-

In [6]:

# 3. Проверка устойчивости (на DS1)

# 1. Загружаем данные заново
df_ds1 = pd.read_csv(datasets['ds1'])
X_ds1 = df_ds1.drop(columns=['sample_id'])

# 2. Переопределяем списки колонок конкретно для этого датасета
num_cols_ds1 = X_ds1.select_dtypes(include=[np.number]).columns.tolist()
cat_cols_ds1 = X_ds1.select_dtypes(exclude=[np.number]).columns.tolist()

# 3. Пересоздаем ColumnTransformer, чтобы он искал 'feat1', 'feat2', а не 'num_0'
preprocessor_ds1 = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols_ds1),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols_ds1)
])

# 4. Трансформируем данные
X_ds1_pre = preprocessor_ds1.fit_transform(X_ds1)

# 5. Запускаем цикл проверки
runs = []
for i in range(5):
    # Используем n_init=1 и разные random_state, чтобы увидеть разницу в разбиениях
    km = KMeans(n_clusters=3, random_state=i*42, n_init=1)
    labels = km.fit_predict(X_ds1_pre)
    runs.append(labels)

# Оценка ARI между первым и вторым запуском (индекс 0 и 1)
ari_score = adjusted_rand_score(runs[0], runs[1])
print(f"\nStability Check (DS1) ARI between two random runs: {ari_score:.4f}")


Stability Check (DS1) ARI between two random runs: 0.5041


In [7]:
# 4. Сохранение JSON артефактов
with open('artifacts/metrics_summary.json', 'w') as f:
    json.dump(all_metrics, f, indent=4)

with open('artifacts/best_configs.json', 'w') as f:
    json.dump(best_configs, f, indent=4)

print("\n--- All tasks completed! Artifacts generated. ---")


--- All tasks completed! Artifacts generated. ---
